In [1]:
import pandas as pd

# Carrega o arquivo CSV
df = pd.read_csv('results_few_shot_quantification.csv')

# 1. Quantidade total de datasets ÚNICOS
total_datasets = df['dataset'].nunique()
print(f"Total de datasets únicos: {total_datasets}")

# 2. Lista com os nomes de cada dataset
nomes_datasets = df['dataset'].unique()
print("Nomes dos datasets:", nomes_datasets)

# 3. Contagem de repetições/experimentos por dataset
contagem_por_dataset = df['dataset'].value_counts()
print(contagem_por_dataset)

Total de datasets únicos: 5
Nomes dos datasets: ['balance.1' 'balance.3' 'breast-cancer' 'cmc.1' 'cmc.2']
dataset
balance.1        3447360
balance.3        3447360
breast-cancer    3447360
cmc.1            3447360
cmc.2             919296
Name: count, dtype: int64


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os
import shutil

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "results_few_shot_quantification.csv"
OS_DIR = "boxplots_binario"

# Métrica utilizada para o ranking
METRIC = "ae"

# Limpa a pasta de saída
shutil.rmtree(OS_DIR, ignore_errors=True)
os.makedirs(OS_DIR, exist_ok=True)


# ============================================================
# PUBLICATION STYLE
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.4,

    "xtick.major.width": 0.0,
    "ytick.major.width": 0.0,
    "xtick.minor.width": 0.0,
    "ytick.minor.width": 0.0,

    "xtick.major.size": 0.0,
    "ytick.major.size": 0.0,
    "xtick.minor.size": 0.0,
    "ytick.minor.size": 0.0,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.top": False,
    "ytick.right": False,

    "axes.grid": True,
    "axes.grid.axis": "y",

    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,
    "grid.color": "#aaaaaa",

    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})


# ============================================================
# LOAD CSV
# ============================================================

print(f"Carregando {CSV_PATH}...")

df_master = pd.read_csv(CSV_PATH)

print(f"Linhas carregadas: {len(df_master):,}")
print(f"Colunas: {df_master.columns.tolist()}")


# ============================================================
# VERIFICAÇÃO DAS COLUNAS
# ============================================================

required_columns = {"dataset", "metodo", "n", METRIC}

missing_columns = required_columns - set(df_master.columns)

if missing_columns:
    raise ValueError(
        f"Colunas ausentes: {missing_columns}\n"
        f"Colunas disponíveis: {df_master.columns.tolist()}"
    )


# ============================================================
# LIMPEZA DOS DADOS
# ============================================================

df_master = df_master.copy()

df_master[METRIC] = pd.to_numeric(
    df_master[METRIC],
    errors="coerce"
)

df_master = df_master.dropna(
    subset=["dataset", "metodo", "n", METRIC]
)


# ============================================================
# PADRONIZA NOMES DOS MÉTODOS
# ============================================================

df_master["modelo"] = df_master["metodo"].replace({

    # SMQ
    "smq_base": "SMQ_BASE",
    "smq_rf": "SMQ_RF",
    "smq_et": "SMQ_ET",
    "smq_gbr": "SMQ_GBR",
    "smq_iso": "SMQ_ISO",

    # Baselines
    "CC": "CC",
    "PCC": "PCC",
    "ACC": "ACC",
    "PACC": "PACC",
    "EMQ": "EMQ",
    "DyS": "DyS",
    "KDEyML": "KDEyML",

    # Few-shot
    "FSNN": "FSNN",
    "FSMM": "FSMM",
    "FSGMM": "FSGMM",
    "FSGPEM": "FSGPEM",
    "FSPL": "FSPL",
})


# ============================================================
# LABELS DOS MÉTODOS
# ============================================================

label_map = {

    # SMQ
    "SMQ_BASE": r"$\mathrm{SMQ}$",
    "SMQ_ISO": r"$\mathrm{SMQ}_{\mathrm{ISO}}$",
    "SMQ_RF": r"$\mathrm{SMQ}_{\mathrm{RF}}$",
    "SMQ_ET": r"$\mathrm{SMQ}_{\mathrm{ET}}$",
    "SMQ_GBR": r"$\mathrm{SMQ}_{\mathrm{GBR}}$",

    # Baselines
    "CC": r"$\mathrm{CC}$",
    "PCC": r"$\mathrm{PCC}$",
    "ACC": r"$\mathrm{ACC}$",
    "PACC": r"$\mathrm{PACC}$",
    "EMQ": r"$\mathrm{EMQ}$",
    "DyS": r"$\mathrm{DyS}$",
    "KDEyML": r"$\mathrm{KDEy}_{\mathrm{ML}}$",

    # Few-shot
    "FSNN": r"$\mathrm{FSNN}$",
    "FSMM": r"$\mathrm{FSMM}$",
    "FSGMM": r"$\mathrm{FSGMM}$",
    "FSGPEM": r"$\mathrm{FSGPEM}$",
    "FSPL": r"$\mathrm{FSPL}$",
}


# ============================================================
# ORDENA OS VALORES DE N
# ============================================================

# Converte tudo para string para tratar "full"
df_master["n_str"] = df_master["n"].astype(str)

# Ordem desejada dos gráficos
n_order_preferred = ["1", "2", "3", "5", "10", "100", "full"]

# Pega apenas valores que realmente existem no CSV
unique_n = df_master["n_str"].unique().tolist()

n_values = [
    n for n in n_order_preferred
    if n in unique_n
]

# Caso apareça algum valor inesperado no CSV
remaining_n = [
    n for n in unique_n
    if n not in n_values
]

n_values.extend(
    sorted(
        remaining_n,
        key=lambda x: (
            x == "full",
            float(x) if x.replace(".", "", 1).isdigit() else float("inf")
        )
    )
)


# ============================================================
# INFORMAÇÕES
# ============================================================

print("\n========================================")
print("CONFIGURAÇÃO")
print("========================================")

print(f"Métrica: {METRIC.upper()}")
print(f"Valores de n encontrados: {n_values}")
print(f"Datasets: {df_master['dataset'].nunique()}")
print(f"Métodos: {df_master['modelo'].nunique()}")

print("========================================\n")


# ============================================================
# LOOP: UM GRÁFICO PARA CADA N
# ============================================================

for n_value in n_values:

    print(f"Gerando gráfico para n = {n_value}...")

    # --------------------------------------------------------
    # FILTRA APENAS O N ATUAL
    # --------------------------------------------------------

    df_n = df_master[
        df_master["n_str"] == str(n_value)
    ].copy()

    # --------------------------------------------------------
    # MÉDIA DO AE
    #
    # Média considerando:
    #
    # dataset + modelo
    #
    # Aqui são agregados batches e repetições,
    # mas APENAS para o n atual.
    # --------------------------------------------------------

    dataset_mean = (
        df_n
        .groupby(["dataset", "modelo"])[METRIC]
        .mean()
        .reset_index()
    )

    # --------------------------------------------------------
    # RANK POR DATASET
    #
    # Menor AE = melhor = Rank 1
    # --------------------------------------------------------

    dataset_mean["rank_dataset"] = (
        dataset_mean
        .groupby("dataset")[METRIC]
        .rank(
            method="average",
            ascending=True
        )
    )

    # --------------------------------------------------------
    # ORDENA MÉTODOS
    #
    # Melhor → pior
    #
    # Critério:
    # menor mediana do ranking
    # --------------------------------------------------------

    ordem_rank = (
        dataset_mean
        .groupby("modelo")["rank_dataset"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    n_models = len(ordem_rank)

    # Labels formatados
    display_labels = [
        label_map.get(modelo, modelo)
        for modelo in ordem_rank
    ]

    # --------------------------------------------------------
    # PALETA
    # --------------------------------------------------------

    palette = sns.color_palette(
        "Spectral",
        n_colors=n_models
    )

    # --------------------------------------------------------
    # TAMANHO DO GRÁFICO
    # --------------------------------------------------------

    fig_width = max(
        7.2,
        n_models * 0.55
    )

    fig, ax = plt.subplots(
        figsize=(fig_width, 3.5)
    )

    # --------------------------------------------------------
    # BOX PLOT
    # --------------------------------------------------------

    sns.boxplot(
        data=dataset_mean,

        x="modelo",
        y="rank_dataset",

        hue="modelo",

        order=ordem_rank,
        hue_order=ordem_rank,

        palette=palette,

        dodge=False,

        legend=False,

        width=0.55,

        linewidth=0.8,

        flierprops=dict(
            marker="o",
            markerfacecolor="none",
            markeredgecolor="#555555",
            markeredgewidth=0.6,
            markersize=3.5,
        ),

        medianprops=dict(
            color="black",
            linewidth=1.4
        ),

        whiskerprops=dict(
            linewidth=0.8
        ),

        capprops=dict(
            linewidth=0.8
        ),

        boxprops=dict(
            linewidth=0.8
        ),

        ax=ax,
    )

    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    if str(n_value).lower() == "full":

        title_n = "Full Training Set"

    else:

        title_n = f"n = {n_value} per class"

    ax.set_title(
        title_n,
        fontsize=11,
        pad=10
    )

    # --------------------------------------------------------
    # AXES
    # --------------------------------------------------------

    ax.set_xlabel(
        "Methods",
        labelpad=6
    )

    ax.set_ylabel(
        "Rank (AE)",
        labelpad=6
    )

    # --------------------------------------------------------
    # LIMITE Y
    # --------------------------------------------------------

    ax.set_ylim(
        0,
        n_models + 1
    )

    ax.yaxis.set_major_locator(
        ticker.MultipleLocator(2)
    )

    ax.yaxis.set_minor_locator(
        ticker.NullLocator()
    )

    # --------------------------------------------------------
    # SPINES
    # --------------------------------------------------------

    for spine in ax.spines.values():

        spine.set_linewidth(0.4)

        spine.set_color(
            "#aaaaaa"
        )

    # --------------------------------------------------------
    # TICKS
    # --------------------------------------------------------

    ax.tick_params(
        which="both",
        length=0
    )

    ax.set_xticks(
        range(n_models)
    )

    ax.set_xticklabels(
        display_labels,

        rotation=40,

        ha="right",

        rotation_mode="anchor"
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    plt.tight_layout()

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    safe_n_value = str(n_value).replace("/", "_")

    nome_arquivo = (
        f"rank_boxplot_binario_n_{safe_n_value}.png"
    )

    output_path = os.path.join(
        OS_DIR,
        nome_arquivo
    )

    plt.savefig(
        output_path,
        dpi=300
    )

    plt.close(fig)

    print(
        f"  Salvo: {output_path}"
    )

    # --------------------------------------------------------
    # PRINT RANKING
    # --------------------------------------------------------

    print(
        f"\nRanking para n = {n_value}"
    )

    for i, modelo in enumerate(
        ordem_rank,
        start=1
    ):

        median_rank = (
            dataset_mean[
                dataset_mean["modelo"] == modelo
            ]["rank_dataset"]
            .median()
        )

        mean_ae = (
            dataset_mean[
                dataset_mean["modelo"] == modelo
            ][METRIC]
            .mean()
        )

        print(
            f"{i:2d}. "
            f"{modelo:<10} "
            f"| Median Rank = {median_rank:.2f} "
            f"| Mean AE = {mean_ae:.6f}"
        )

    print()


# ============================================================
# FINAL
# ============================================================

print("========================================")
print("CONCLUÍDO!")
print("========================================")

print(f"Todos os gráficos foram salvos em:")
print(OS_DIR)

print("\nGráficos gerados:")

for filename in sorted(os.listdir(OS_DIR)):
    print(f" - {filename}")

Carregando results_few_shot_quantification.csv...
Linhas carregadas: 14,708,736
Colunas: ['dataset', 'n', 'metodo', 'ae', 'rae', 'repeticao']

CONFIGURAÇÃO
Métrica: AE
Valores de n encontrados: ['1', '2', '3', '5', '10', '100', 'full']
Datasets: 5
Métodos: 17

Gerando gráfico para n = 1...
  Salvo: boxplots_binario/rank_boxplot_binario_n_1.png

Ranking para n = 1
 1. FSMM       | Median Rank = 1.00 | Mean AE = 0.200701
 2. CC         | Median Rank = 3.00 | Mean AE = 0.217334
 3. FSNN       | Median Rank = 4.00 | Mean AE = 0.216949
 4. FSGPEM     | Median Rank = 5.00 | Mean AE = 0.221559
 5. FSPL       | Median Rank = 5.00 | Mean AE = 0.220810
 6. PCC        | Median Rank = 6.00 | Mean AE = 0.227728
 7. FSGMM      | Median Rank = 7.00 | Mean AE = 0.242397
 8. SMQ_ET     | Median Rank = 7.00 | Mean AE = 0.231278
 9. SMQ_GBR    | Median Rank = 8.00 | Mean AE = 0.241229
10. SMQ_RF     | Median Rank = 9.00 | Mean AE = 0.238750
11. SMQ_BASE   | Median Rank = 11.00 | Mean AE = 0.245747
12. SM

In [ ]:
testar outras tecnicas de calibração multiclasse no smq iso tipo temperature scaling e dirthclet 
hibrido vs outros metodos de oversampling